# Implementation — Pulkkinen et al. (2017): GIC Pipeline / 구현 — Pulkkinen et al. (2017): GIC 파이프라인

**Paper:** Pulkkinen, A. et al. *Geomagnetically induced currents: Science, engineering, and applications readiness.* Space Weather, 15(7), 828–856 (2017). DOI: 10.1002/2016SW001501

## Goal / 목표
**English** Reproduce the canonical GIC pipeline: synthetic ground magnetic field B(t) → time derivative dB/dt → induced geoelectric field E(t) via the plane-wave method → GIC in a single transmission line via the simple I = E·L/R model → hazard map concept (99th-percentile dB/dt over a region).

**Korean** 정규 GIC 파이프라인을 재현한다: 합성 지면 자기장 B(t) → 시간 미분 dB/dt → 평면파 방법으로 유도된 지전기장 E(t) → 단일 송전선 GIC (I = E·L/R) → 지역 단위 99 백분위 dB/dt 기반 위험 지도 개념.

## Part 1 — Setup and synthetic ground magnetic field / 셋업 및 합성 지면 자기장

**English** We model B(t) as the superposition of a slow main-phase decay (storm-time ring current) and a rapid substorm onset transient. The latter dominates dB/dt.

**Korean** B(t)를 느린 주(메인) 위상 감쇠(storm-time ring current)와 빠른 substorm onset 과도성분의 중첩으로 모델한다. 후자가 dB/dt를 지배한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Time grid: 6 hours sampled at 1 Hz.
FS = 1.0  # sampling frequency [Hz]
T_TOTAL = 6 * 3600  # total duration [s]
t = np.arange(0, T_TOTAL, 1.0 / FS)
N = t.size


def synth_b_field(t: np.ndarray) -> np.ndarray:
    """Generate a synthetic horizontal ground magnetic perturbation [nT].

    Combines a slow ring-current main-phase decay with a substorm onset
    transient and a colored-noise micropulsation background.

    Args:
        t: Time array in seconds.

    Returns:
        Magnetic perturbation B(t) in nanotesla.
    """
    # Storm main phase: -300 nT bay over ~3 hours.
    main_phase = -300.0 * np.exp(-((t - 7200) ** 2) / (2 * 3600 ** 2))
    # Substorm onset: sharp 400 nT decrease over ~120 s near t = 2.5 h.
    onset_t0 = 9000.0
    onset = -400.0 / (1 + np.exp((t - onset_t0) / 30.0))
    # Colored Pi2 micropulsations (~0.01 Hz) with random phase.
    micro = 15.0 * np.sin(2 * np.pi * 0.01 * t + np.random.randn())
    # White noise floor.
    noise = 5.0 * np.random.randn(t.size)
    return main_phase + onset + micro + noise


B = synth_b_field(t)
print(f"B range: [{B.min():.1f}, {B.max():.1f}] nT, samples = {N}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t / 3600, B, lw=0.6)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("B [nT]")
ax.set_title("Synthetic ground magnetic perturbation / 합성 지면 자기장")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2 — dB/dt time derivative / dB/dt 시간 미분

**English** GIC physics is driven by dB/dt rather than B itself. We compute it via centered finite differences and integrate the result over time using `np.trapezoid` to confirm round-trip consistency.

**Korean** GIC 물리는 B 자체가 아닌 dB/dt가 구동한다. 중앙 차분으로 계산하고 `np.trapezoid`로 시간 적분하여 왕복 일관성을 검증한다.

In [ ]:
dBdt = np.gradient(B, 1.0 / FS)  # nT/s

# Round-trip check: integrate dB/dt back to B using trapezoidal rule.
B_reconstructed = np.trapezoid(
    np.vstack([np.zeros_like(dBdt), dBdt]).T, dx=1.0 / FS, axis=1
)
# Simpler equivalent path: cumulative trapezoid via running np.trapezoid.
B_running = np.array([np.trapezoid(dBdt[: i + 1], dx=1.0 / FS) for i in range(N)])
B_running += B[0]  # add integration constant.

rms_err = np.sqrt(np.mean((B_running - B) ** 2))
print(f"Reconstruction RMS error: {rms_err:.3f} nT")

fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axs[0].plot(t / 3600, B, lw=0.6)
axs[0].set_ylabel("B [nT]")
axs[0].grid(alpha=0.3)
axs[1].plot(t / 3600, dBdt, lw=0.6, color="crimson")
axs[1].set_ylabel("dB/dt [nT/s]")
axs[1].set_xlabel("Time [hours]")
axs[1].grid(alpha=0.3)
axs[0].set_title("B and dB/dt / B와 dB/dt")
plt.tight_layout()
plt.show()

peak_idx = int(np.argmax(np.abs(dBdt)))
print(f"Peak |dB/dt| = {np.abs(dBdt[peak_idx]):.2f} nT/s at t = {t[peak_idx] / 3600:.2f} h")

## Part 3 — Geoelectric field via plane-wave method / 평면파 방법으로 지전기장

**English** For a uniform half-space of conductivity σ, the surface impedance is
$$Z(\omega) = \sqrt{i\omega\mu_0/\sigma}.$$
We Fourier-transform B(t), multiply by the transfer function $\zeta(\omega) = Z(\omega)/\mu_0$, and inverse-transform to obtain E(t). Two conductivities are compared: a conductive sediment ($\sigma=10^{-2}$ S/m) and a resistive shield ($\sigma=10^{-4}$ S/m).

**Korean** 균일 반무한 매체(전도도 σ)의 지표 임피던스는 위의 식과 같다. B(t)를 푸리에 변환하고 전달함수를 곱한 뒤 역변환하여 E(t)를 구한다. 전도성 퇴적층과 저항성 순상지 두 경우를 비교한다.

In [ ]:
MU0 = 4.0 * np.pi * 1e-7  # vacuum permeability [H/m]


def plane_wave_efield(b_nT: np.ndarray, fs: float, sigma: float) -> np.ndarray:
    """Compute geoelectric field from B via the plane-wave method.

    Implements E(ω) = ζ(ω) · B(ω) with ζ(ω) = sqrt(iωμ0/σ) / μ0,
    using rfft/irfft for real input.

    Args:
        b_nT: Magnetic field time series in nanotesla.
        fs: Sampling frequency in Hz.
        sigma: Half-space conductivity in S/m.

    Returns:
        Geoelectric field E(t) in V/m.
    """
    n = b_nT.size
    b_si = b_nT * 1e-9  # nT → T
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    omega = 2.0 * np.pi * freqs
    # Transfer function ζ(ω) for half-space; DC value forced to zero (no induction at ω=0).
    z = np.sqrt(1j * omega * MU0 / sigma)
    zeta = np.zeros_like(z, dtype=complex)
    zeta[1:] = z[1:] / MU0
    b_spec = np.fft.rfft(b_si)
    e_spec = zeta * b_spec
    e_t = np.fft.irfft(e_spec, n=n)
    return e_t


sigma_sediment = 1e-2  # conductive sediments [S/m]
sigma_shield = 1e-4  # resistive Precambrian shield [S/m]

E_sed = plane_wave_efield(B, FS, sigma_sediment)
E_shi = plane_wave_efield(B, FS, sigma_shield)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t / 3600, E_sed * 1e3, label=f"σ=10⁻² S/m (sediment)", lw=0.7)
ax.plot(t / 3600, E_shi * 1e3, label=f"σ=10⁻⁴ S/m (shield)", lw=0.7, color="darkred")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("E [mV/m] = [V/km]")
ax.set_title("Induced geoelectric field / 유도 지전기장")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak |E| sediment: {np.max(np.abs(E_sed)) * 1e3:.3f} V/km")
print(f"Peak |E| shield  : {np.max(np.abs(E_shi)) * 1e3:.3f} V/km")
print(f"Amplification ratio: {np.max(np.abs(E_shi)) / np.max(np.abs(E_sed)):.1f}x")

## Part 4 — GIC in a transmission line / 송전선 GIC

**English** Using the simple model $I = E\cdot L / R$, we estimate the GIC in a 200 km line of total resistance 5 Ω (line + ground return) for both Earth conductivity scenarios.

**Korean** 단순 모델 $I = E\cdot L / R$로, 길이 200 km, 합성 저항 5 Ω (라인 + 대지 귀로)인 송전선에 흐르는 GIC를 두 전도성 시나리오에서 추정한다.

In [ ]:
L_LINE = 200e3  # line length [m]
R_LINE = 5.0  # combined resistance [Ω]


def gic_simple_model(e_field: np.ndarray, length_m: float, resistance_ohm: float) -> np.ndarray:
    """Compute GIC via the simple I = E·L/R model.

    Args:
        e_field: Geoelectric field along the line [V/m].
        length_m: Line length [m].
        resistance_ohm: Total line + ground return resistance [Ω].

    Returns:
        GIC time series [A].
    """
    return e_field * length_m / resistance_ohm


I_sed = gic_simple_model(E_sed, L_LINE, R_LINE)
I_shi = gic_simple_model(E_shi, L_LINE, R_LINE)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t / 3600, I_sed, label="sediment Earth", lw=0.7)
ax.plot(t / 3600, I_shi, label="shield Earth", lw=0.7, color="darkred")
ax.axhline(100, color="k", ls="--", alpha=0.4, label="100 A reference")
ax.axhline(-100, color="k", ls="--", alpha=0.4)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("GIC [A]")
ax.set_title("GIC in 200 km transmission line / 200 km 송전선 GIC")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak GIC sediment: {np.max(np.abs(I_sed)):.1f} A")
print(f"Peak GIC shield  : {np.max(np.abs(I_shi)):.1f} A")

## Part 5 — Hazard map: 99th-percentile dB/dt over a region / 위험 지도: 지역별 99 백분위 dB/dt

**English** A simplified hazard map is built by simulating B(t) on a 10×10 spatial grid with a latitude-dependent peak amplitude (auroral oval near 60° magnetic) and computing the 99th percentile of |dB/dt| at each cell. This mimics the operational construction of dB/dt hazard maps.

**Korean** 10×10 공간 격자에서 위도 의존적으로 첨두 진폭을 변화시키며(자기위도 60° 근처 오로라 타원) B(t)를 시뮬레이션하고, 각 셀의 |dB/dt| 99 백분위를 계산한다. 운영용 dB/dt 위험 지도의 축소판이다.

In [ ]:
NX, NY = 10, 10
lat_grid = np.linspace(40.0, 70.0, NY)  # geomagnetic latitude rows
lon_grid = np.linspace(-100.0, -70.0, NX)

rng = np.random.default_rng(7)
hazard = np.zeros((NY, NX))

for j, mlat in enumerate(lat_grid):
    # Auroral-oval-like envelope peaking at 60° magnetic latitude.
    envelope = np.exp(-((mlat - 60.0) ** 2) / (2 * 5.0 ** 2))
    for i in range(NX):
        b_local = synth_b_field(t) * (0.3 + 1.7 * envelope) + 20.0 * rng.standard_normal(N)
        dbdt_local = np.gradient(b_local, 1.0 / FS)
        hazard[j, i] = np.percentile(np.abs(dbdt_local), 99)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(
    hazard,
    origin="lower",
    extent=[lon_grid[0], lon_grid[-1], lat_grid[0], lat_grid[-1]],
    aspect="auto",
    cmap="viridis",
)
cb = plt.colorbar(im, ax=ax)
cb.set_label("99th percentile |dB/dt| [nT/s]")
ax.set_xlabel("Geomagnetic longitude [°]")
ax.set_ylabel("Geomagnetic latitude [°]")
ax.set_title("Synthetic dB/dt hazard map / 합성 dB/dt 위험 지도")
plt.tight_layout()
plt.show()

peak_lat = lat_grid[int(np.argmax(hazard.max(axis=1)))]
print(f"Peak hazard at magnetic latitude ≈ {peak_lat:.1f}°")
print(
    f"Range of 99th-percentile |dB/dt|: [{hazard.min():.2f}, {hazard.max():.2f}] nT/s"
)

## Part 6 — Summary and discussion / 요약 및 논의

**English**
1. The plane-wave method correctly amplifies E in resistive (shield) Earth by an order of magnitude vs. conductive (sediment) Earth, reproducing the qualitative finding of Pulkkinen et al. (2017).
2. Peak GIC scales linearly with E and L; for typical mid-latitude lines the simulated peaks fall in the 30–300 A range observed during major storms.
3. The synthetic hazard map peaks at ≈60° magnetic latitude — consistent with the subauroral high-risk zone identified in the paper.
4. The implementation uses the simplified scalar plane-wave method; the paper's full formulation employs the **2×2 surface impedance tensor** (Eq. 2, p. 841) which captures 3-D induction effects, and the **Lehtinen-Pirjola network equations (Eqs. 3–6, p. 843)** for full grid topology.
5. Per the paper's ARL framework (Figure 2), the geoelectric-field link (Link C) and the GIC computation link (Link B) are at **ARL 9** — i.e., this implementation reproduces operationally mature parts of the chain. The weakest links are the upper coronal/solar-wind drivers (Links G, F at ARL 4) and the engineering-impact link (Link A at ARL 6).
6. Limitations: (a) uniform half-space ignores 3-D Earth heterogeneity and underestimates regional contrasts; (b) the simple I = E·L/R model neglects network coupling captured by the Lehtinen-Pirjola matrix equation; (c) synthetic B is not a real storm time series.

**Korean**
1. 평면파 방법은 저항성 순상지에서 전도성 퇴적지 대비 E를 한 자릿수 증폭시켜 Pulkkinen et al. (2017)의 정성적 결과를 재현한다.
2. 첨두 GIC는 E와 L에 선형 비례하며, 일반 중위도 송전선에서 시뮬레이션 첨두는 주요 폭풍 시 관측되는 30–300 A 범위 내에 있다.
3. 합성 위험 지도는 자기위도 ≈60°에서 최대이며 논문이 지적한 부오로라 고위험 영역과 일치한다.
4. 본 구현은 단순화된 스칼라 평면파 방법을 사용한다; 논문의 전체 정식화는 3D 유도 효과를 포착하는 **2×2 지표 임피던스 텐서**(Eq. 2, p. 841)와 전체 송전망 토폴로지를 위한 **Lehtinen-Pirjola 네트워크 방정식**(Eqs. 3–6, p. 843)을 사용한다.
5. 논문의 ARL 체계(Figure 2)에 따르면, 지전기장 링크(Link C)와 GIC 계산 링크(Link B)는 **ARL 9** — 즉, 본 구현은 체인 중 운영 성숙도가 가장 높은 부분을 재현한다. 가장 취약한 링크는 상부 코로나/태양풍 구동(Links G, F: ARL 4)과 공학 임팩트 링크(Link A: ARL 6)이다.
6. 한계: (a) 균일 반무한 매체는 3D 지구 비균일성과 지역 대비를 저평가; (b) 단순 I = E·L/R 모델은 Lehtinen-Pirjola 행렬 방정식이 포착하는 네트워크 결합을 무시; (c) 합성 B는 실제 폭풍 시계열이 아님.